# Lab 11: Security, Compliance & Logging

## Overview
In this lab you will implement security best practices for Amazon Bedrock. You will create least-privilege IAM policies for different personas, understand VPC endpoints (PrivateLink) for private access, configure KMS encryption for model customization, enable CloudTrail audit logging, set up model invocation logging to capture prompts and responses, and implement cost governance through tagging and budgets.

## Learning Objectives
- Create least-privilege IAM policies for Bedrock with persona-based access control
- Understand VPC endpoints (PrivateLink) for private network access to Bedrock
- Configure KMS encryption for model customization and invocation logging
- Enable CloudTrail audit logging and query Bedrock API events
- Set up model invocation logging to capture inputs and outputs
- Implement cost tagging and budgets for multi-team GenAI governance

## Exam Domain
**Security, Compliance & Governance (20%)** — This lab covers Task 4.1 (implement security controls for AI/ML workloads) and Task 4.3 (ensure compliance and governance of AI/ML solutions), focusing on IAM policies, network isolation, encryption, audit logging, and cost governance for Bedrock deployments.

## Architecture Diagram
![Lab 11 Architecture](../assets/diagrams/lab-11-security-governance.png)

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3, json, os, time
from datetime import datetime, timedelta

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

iam = session.client("iam")
ec2 = session.client("ec2")
kms = session.client("kms")
cloudtrail = session.client("cloudtrail")
bedrock = session.client("bedrock")
bedrock_runtime = session.client("bedrock-runtime")
cloudwatch = session.client("logs")
sts = session.client("sts")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"

print(f"Account:      {ACCOUNT_ID}")
print(f"Region:       {REGION}")
print(f"Bedrock role: {BEDROCK_ROLE_ARN}")
print(f"S3 bucket:    {BUCKET}")

## A. IAM Least-Privilege Policies

The principle of least privilege means granting only the permissions required to perform a specific task. For Bedrock, different personas need different levels of access. In this section we define three IAM policies — Developer, Data Scientist, and Admin — and analyze what each permits.

**Why it matters for the exam:** IAM policy design is a core topic in the Security domain. You must understand which Bedrock actions map to which job functions and how to scope `Resource` ARNs to limit blast radius.

In [ ]:
# Policy 1: Developer — invoke models only (no management, no customization)
developer_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "BedrockInvokeOnly",
        "Effect": "Allow",
        "Action": [
            "bedrock:InvokeModel",
            "bedrock:InvokeModelWithResponseStream"
        ],
        "Resource": "arn:aws:bedrock:us-east-1::foundation-model/*"
    }]
}

print("Developer Policy — invoke foundation models only")
print(json.dumps(developer_policy, indent=2))
print("\nThis policy allows a developer to call any foundation model")
print("but NOT create custom models, manage guardrails, or change settings.")

In [ ]:
# Policy 2: Data Scientist — invoke + fine-tune + manage custom models
data_scientist_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockInvoke",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": "arn:aws:bedrock:us-east-1::foundation-model/*"
        },
        {
            "Sid": "BedrockCustomization",
            "Effect": "Allow",
            "Action": [
                "bedrock:CreateModelCustomizationJob",
                "bedrock:GetModelCustomizationJob",
                "bedrock:ListModelCustomizationJobs"
            ],
            "Resource": "*"
        },
        {
            "Sid": "BedrockCustomModels",
            "Effect": "Allow",
            "Action": [
                "bedrock:CreateProvisionedModelThroughput",
                "bedrock:GetCustomModel"
            ],
            "Resource": "*"
        }
    ]
}

print("Data Scientist Policy — invoke + fine-tune + custom models")
print(json.dumps(data_scientist_policy, indent=2))
print("\nAdds model customization (fine-tuning) and provisioned throughput")
print("to the developer permissions. Still cannot manage guardrails or logging.")

In [ ]:
# Policy 3: Admin — full Bedrock access (still scoped to Bedrock only)
admin_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "BedrockFullAccess",
        "Effect": "Allow",
        "Action": "bedrock:*",
        "Resource": "*"
    }]
}

print("Admin Policy — full Bedrock access")
print(json.dumps(admin_policy, indent=2))
print("\nThis grants ALL Bedrock actions: invoke, customize, manage guardrails,")
print("configure logging, manage knowledge bases, agents, and more.")
print("\nIMPORTANT: Even though this uses 'bedrock:*', it is still scoped to the")
print("Bedrock service only. It does NOT grant access to S3, IAM, KMS, or other")
print("services. The admin would also need separate policies for supporting services.")

In [ ]:
# Compare the three policies side by side
policies = {
    "Developer": developer_policy,
    "Data Scientist": data_scientist_policy,
    "Admin": admin_policy
}

print("Policy Comparison — Actions per Persona")
print("=" * 65)
for persona, policy in policies.items():
    actions = []
    for stmt in policy["Statement"]:
        act = stmt["Action"]
        if isinstance(act, list):
            actions.extend(act)
        else:
            actions.append(act)
    print(f"\n{persona}:")
    for action in actions:
        print(f"  - {action}")

print("\n" + "=" * 65)
print("\nIAM Policy Evaluation Logic:")
print("1. By default, all actions are DENIED (implicit deny)")
print("2. An explicit ALLOW overrides the implicit deny")
print("3. An explicit DENY always overrides any ALLOW")
print("4. Policies are evaluated across all attached policies (union)")
print("\nTo validate policies, use the IAM Policy Simulator:")
print("  https://policysim.aws.amazon.com/")
print("  Or programmatically via iam.simulate_principal_policy()")

## B. VPC Endpoints (PrivateLink)

By default, calls to Amazon Bedrock traverse the public internet. A **VPC endpoint** (powered by AWS PrivateLink) creates a private connection between your VPC and the Bedrock service without requiring an internet gateway, NAT device, or VPN connection.

**Why VPC endpoints matter:**
- **No internet exposure** — traffic stays on the AWS private network, never crossing the public internet
- **Compliance** — required for regulated workloads (HIPAA, PCI, FedRAMP) that prohibit internet-bound API calls
- **Data residency** — ensures prompts and responses remain within the AWS network
- **Network security** — can be combined with security groups and VPC endpoint policies for fine-grained access control

In [ ]:
# List available Bedrock VPC endpoint services in the region
bedrock_services = ec2.describe_vpc_endpoint_services(
    Filters=[{"Name": "service-name", "Values": [f"*bedrock*"]}]
)

print("Available Bedrock VPC Endpoint Services")
print("=" * 55)
for svc in bedrock_services.get("ServiceNames", []):
    print(f"  {svc}")

print(f"\nTotal: {len(bedrock_services.get('ServiceNames', []))} services")
print("\nKey services:")
print(f"  com.amazonaws.{REGION}.bedrock          — Bedrock control plane (manage models, guardrails)")
print(f"  com.amazonaws.{REGION}.bedrock-runtime   — Bedrock runtime (invoke models)")
print(f"  com.amazonaws.{REGION}.bedrock-agent     — Bedrock agents")
print(f"  com.amazonaws.{REGION}.bedrock-agent-runtime — Bedrock agent runtime (invoke agents)")

In [ ]:
# Conceptual: Creating a VPC endpoint for Bedrock Runtime
# This requires an existing VPC, subnets, and security groups.
# The code below shows the exact API call — commented out to avoid
# creating infrastructure in this lab environment.

# vpc_endpoint = ec2.create_vpc_endpoint(
#     VpcEndpointType="Interface",
#     VpcId="vpc-xxx",
#     ServiceName=f"com.amazonaws.{REGION}.bedrock-runtime",
#     SubnetIds=["subnet-xxx"],
#     SecurityGroupIds=["sg-xxx"],
#     PrivateDnsEnabled=True
# )
# print(f"Created VPC Endpoint: {vpc_endpoint['VpcEndpoint']['VpcEndpointId']}")

# Conceptual: VPC endpoint policy to restrict which models can be invoked
vpc_endpoint_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "AllowBedrockInvoke",
        "Effect": "Allow",
        "Principal": "*",
        "Action": [
            "bedrock:InvokeModel",
            "bedrock:InvokeModelWithResponseStream"
        ],
        "Resource": "arn:aws:bedrock:us-east-1::foundation-model/*"
    }]
}

print("VPC Endpoint Policy (restricts access at the network level):")
print(json.dumps(vpc_endpoint_policy, indent=2))

print("\nWith PrivateDnsEnabled=True, existing code using the standard")
print("Bedrock endpoint (bedrock-runtime.us-east-1.amazonaws.com) will")
print("automatically route through the VPC endpoint — no code changes needed.")

print("\nWhen to use VPC endpoints:")
print("  - Regulated industries (healthcare, finance) with compliance requirements")
print("  - Workloads processing PII or sensitive data in prompts")
print("  - Organizations with strict no-internet policies for production workloads")
print("  - Defense-in-depth security posture alongside IAM policies")

## C. Encryption with KMS

Amazon Bedrock encrypts data at rest and in transit by default. However, for model customization jobs, invocation logging, and knowledge bases, you can provide your own **AWS KMS customer managed key (CMK)** for additional control.

**Encryption in Bedrock:**
- **In transit** — all API calls use TLS 1.2+ (automatic, no configuration needed)
- **At rest (default)** — AWS-managed keys encrypt stored data (training data, custom models, logs)
- **At rest (CMK)** — customer managed KMS keys give you full control over key rotation, access policies, and audit trails via CloudTrail

In [ ]:
# Create a KMS customer managed key for Bedrock encryption
key = kms.create_key(
    Description="Bedrock lab encryption key",
    KeyUsage="ENCRYPT_DECRYPT",
    Origin="AWS_KMS",
    Tags=[
        {"TagKey": "Project", "TagValue": "genai-lab"},
        {"TagKey": "Lab", "TagValue": "11-security-governance"}
    ]
)

KEY_ID = key["KeyMetadata"]["KeyId"]
KEY_ARN = key["KeyMetadata"]["Arn"]

print(f"KMS Key ID:  {KEY_ID}")
print(f"KMS Key ARN: {KEY_ARN}")
print(f"Key State:   {key['KeyMetadata']['KeyState']}")
print(f"Key Usage:   {key['KeyMetadata']['KeyUsage']}")

In [ ]:
# Create an alias for easier reference
kms.create_alias(
    AliasName="alias/genai-lab-bedrock",
    TargetKeyId=KEY_ID
)
print(f"Created alias: alias/genai-lab-bedrock -> {KEY_ID}")

# Show how KMS keys are referenced in Bedrock API calls
print("\n" + "=" * 65)
print("How KMS keys are used with Bedrock:")
print("=" * 65)

print("\n1. Model Customization Job (fine-tuning):")
print("""   bedrock.create_model_customization_job(
       ...
       customModelKmsKeyId=KEY_ARN,   # <-- encrypts the custom model
       ...
   )""")

print("\n2. Invocation Logging (encrypt logs at rest):")
print("""   bedrock.put_model_invocation_logging_configuration(
       loggingConfig={
           "cloudWatchConfig": {
               ...
               "largeDataDeliveryS3Config": {
                   "bucketName": "my-bucket",
                   "keyPrefix": "logs/",
                   "s3EncryptionConfiguration": {
                       "kmsKeyId": KEY_ARN   # <-- encrypts log data in S3
                   }
               }
           }
       }
   )""")

print("\n3. Knowledge Base (encrypt vector store data):")
print("""   bedrock.create_knowledge_base(
       ...
       serverSideEncryptionConfiguration={
           "kmsKeyArn": KEY_ARN   # <-- encrypts knowledge base data
       }
   )""")

print(f"\nYour key ARN for reference: {KEY_ARN}")

## D. CloudTrail Logging

AWS CloudTrail records API calls made to your AWS account. Every Bedrock API call — `InvokeModel`, `CreateGuardrail`, `PutModelInvocationLoggingConfiguration`, etc. — generates a CloudTrail event.

**What CloudTrail captures:**
- Which API was called (`EventName`)
- Who called it (`Username`, IAM principal)
- When it was called (`EventTime`)
- Where the call came from (`SourceIPAddress`)
- Request parameters and response elements

**What CloudTrail does NOT capture:**
- The actual prompts sent to models
- The model responses/completions
- Token counts or latency metrics

For prompt and response logging, you need **Model Invocation Logging** (Section E).

In [ ]:
# Query CloudTrail for recent Bedrock API calls
events = cloudtrail.lookup_events(
    LookupAttributes=[{
        "AttributeKey": "EventSource",
        "AttributeValue": "bedrock.amazonaws.com"
    }],
    MaxResults=10
)

print("Recent Bedrock API Events (from CloudTrail)")
print("=" * 75)

if events["Events"]:
    for event in events["Events"]:
        event_time = event["EventTime"].strftime("%Y-%m-%d %H:%M:%S")
        event_name = event["EventName"]
        username = event.get("Username", "N/A")
        print(f"  {event_time} | {event_name:<45} | {username}")
else:
    print("  No recent Bedrock events found in CloudTrail.")
    print("  (Events may take 5-15 minutes to appear after API calls.)")

print(f"\nTotal events returned: {len(events['Events'])}")

In [ ]:
# Inspect a CloudTrail event in detail (if available)
if events["Events"]:
    sample_event = json.loads(events["Events"][0]["CloudTrailEvent"])
    print("Sample CloudTrail Event Detail")
    print("=" * 55)
    print(f"  Event Name:      {sample_event.get('eventName')}")
    print(f"  Event Source:    {sample_event.get('eventSource')}")
    print(f"  Event Time:      {sample_event.get('eventTime')}")
    print(f"  AWS Region:      {sample_event.get('awsRegion')}")
    print(f"  Source IP:       {sample_event.get('sourceIPAddress')}")
    print(f"  User Agent:      {sample_event.get('userAgent', 'N/A')[:80]}")
    print(f"  User Identity:   {sample_event.get('userIdentity', {}).get('type', 'N/A')}")
    print(f"  Request Params:  {json.dumps(sample_event.get('requestParameters', {}))[:120]}...")
else:
    print("No events available to inspect.")
    print("Run a Bedrock API call in another lab, wait 5-15 minutes, then re-run.")

print("\nCloudTrail answers: WHO did WHAT, WHEN, and from WHERE.")
print("It does NOT log the content of prompts or model responses.")

## E. Model Invocation Logging

Model invocation logging captures the **actual prompts and responses** sent to and from Bedrock models. This is separate from CloudTrail, which only records API metadata.

| Feature | CloudTrail | Invocation Logging |
|---------|-----------|-------------------|
| What is logged | API call metadata (who, when, what action) | Prompt text, model response, token counts |
| Automatically enabled | Yes (for management events) | No — must be explicitly configured |
| Destination | CloudTrail S3 bucket / CloudWatch Logs | CloudWatch Logs and/or S3 bucket |
| Use case | Security audit, compliance | Content audit, debugging, quality monitoring |
| Cost | Free (management events) | CloudWatch Logs + S3 storage costs |

In [ ]:
# Check current invocation logging configuration
try:
    current_config = bedrock.get_model_invocation_logging_configuration()
    logging_config = current_config.get("loggingConfig", {})
    if logging_config:
        print("Current Invocation Logging Configuration:")
        print(json.dumps(logging_config, indent=2, default=str))
    else:
        print("No invocation logging currently configured.")
except Exception as e:
    print(f"Could not retrieve logging config: {e}")

In [ ]:
# Enable model invocation logging — sends prompts and responses to
# CloudWatch Logs and S3 for audit and debugging
try:
    bedrock.put_model_invocation_logging_configuration(
        loggingConfig={
            "cloudWatchConfig": {
                "logGroupName": "/aws/bedrock/model-invocations",
                "roleArn": BEDROCK_ROLE_ARN,
                "largeDataDeliveryS3Config": {
                    "bucketName": BUCKET,
                    "keyPrefix": "invocation-logs/"
                }
            },
            "s3Config": {
                "bucketName": BUCKET,
                "keyPrefix": "invocation-logs/"
            },
            "textDataDeliveryEnabled": True,
            "imageDataDeliveryEnabled": False,
            "embeddingDataDeliveryEnabled": False
        }
    )
    print("Model invocation logging ENABLED")
    print(f"  CloudWatch: /aws/bedrock/model-invocations")
    print(f"  S3:         s3://{BUCKET}/invocation-logs/")
    print(f"  Text data:  Enabled")
    print(f"  Image data: Disabled")
    print(f"  Embedding:  Disabled")
except Exception as e:
    print(f"Could not enable invocation logging: {e}")
    print("\nThis may fail if the IAM role lacks CloudWatch/S3 permissions.")
    print("Required role permissions: logs:CreateLogGroup, logs:CreateLogStream,")
    print("logs:PutLogEvents, s3:PutObject")

In [ ]:
# Make a test invocation so there's something to log
test_response = bedrock_runtime.converse(
    modelId="meta.llama3-8b-instruct-v1:0",
    messages=[{"role": "user", "content": [{"text": "What is IAM in AWS? Reply in one sentence."}]}],
    inferenceConfig={"maxTokens": 100, "temperature": 0.1}
)

output_text = test_response["output"]["message"]["content"][0]["text"]
input_tokens = test_response["usage"]["inputTokens"]
output_tokens = test_response["usage"]["outputTokens"]

print("Test Invocation (for logging verification)")
print("=" * 55)
print(f"Model:         meta.llama3-8b-instruct-v1:0")
print(f"Prompt:        What is IAM in AWS? Reply in one sentence.")
print(f"Response:      {output_text}")
print(f"Input tokens:  {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"\nThis invocation should appear in the invocation logs within a few minutes.")

In [ ]:
# Check CloudWatch Logs for invocation log entries
# Note: logs may take 1-5 minutes to appear after invocation
LOG_GROUP = "/aws/bedrock/model-invocations"

try:
    # Check if log group exists
    log_groups = cloudwatch.describe_log_groups(
        logGroupNamePrefix=LOG_GROUP
    )

    if log_groups["logGroups"]:
        print(f"Log group exists: {LOG_GROUP}")

        # Get recent log streams
        streams = cloudwatch.describe_log_streams(
            logGroupName=LOG_GROUP,
            orderBy="LastEventTime",
            descending=True,
            limit=3
        )

        if streams["logStreams"]:
            print(f"Found {len(streams['logStreams'])} log stream(s)\n")

            # Read events from the most recent stream
            latest_stream = streams["logStreams"][0]["logStreamName"]
            log_events = cloudwatch.get_log_events(
                logGroupName=LOG_GROUP,
                logStreamName=latest_stream,
                limit=5
            )

            print(f"Latest log stream: {latest_stream}")
            print("-" * 55)
            for event in log_events["events"]:
                timestamp = datetime.fromtimestamp(event["timestamp"] / 1000)
                message = json.loads(event["message"])
                print(f"  Time:    {timestamp}")
                print(f"  Model:   {message.get('modelId', 'N/A')}")
                print(f"  Tokens:  in={message.get('inputTokenCount', 'N/A')}, out={message.get('outputTokenCount', 'N/A')}")
                print()
        else:
            print("No log streams yet — logs may take a few minutes to appear.")
    else:
        print(f"Log group {LOG_GROUP} not found.")
        print("It will be created automatically when the first log is delivered.")
except Exception as e:
    print(f"Could not query logs: {e}")
    print("Logs may take a few minutes to appear after enabling invocation logging.")

## F. Cost Governance

Cost governance for GenAI workloads combines **resource tagging** (for cost allocation and tracking) with **AWS Budgets** (for alerting when spend exceeds thresholds). Tags enable you to attribute Bedrock costs to specific teams, projects, or environments.

**Tagging strategy for multi-team GenAI:**
- `Project` — which project or application (e.g., "customer-support-bot")
- `Environment` — dev, staging, production
- `Team` — which team owns the resource
- `CostCenter` — for finance/chargeback alignment

In [ ]:
# Demonstrate resource tagging for Bedrock resources
# Note: tag_resource requires a valid resource ARN — we'll show the pattern
# and tag our KMS key as a concrete example

# Tag the KMS key we created earlier
kms.tag_resource(
    KeyId=KEY_ID,
    Tags=[
        {"TagKey": "Project", "TagValue": "genai-lab"},
        {"TagKey": "Environment", "TagValue": "development"},
        {"TagKey": "Team", "TagValue": "ml-platform"},
        {"TagKey": "CostCenter", "TagValue": "CC-1234"}
    ]
)
print(f"Tagged KMS key {KEY_ID} with cost allocation tags")

# Show how Bedrock resources are tagged (conceptual — requires existing resources)
print("\nBedrock Resource Tagging Examples:")
print("=" * 55)

print("""
# Tag a guardrail
bedrock.tag_resource(
    resourceARN=f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:guardrail/abc123",
    tags=[
        {"key": "Project", "value": "genai-lab"},
        {"key": "Environment", "value": "development"},
        {"key": "Team", "value": "ml-platform"},
        {"key": "CostCenter", "value": "CC-1234"}
    ]
)

# Tag a custom model
bedrock.tag_resource(
    resourceARN=f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:custom-model/my-model",
    tags=[
        {"key": "Project", "value": "customer-support"},
        {"key": "Environment", "value": "production"},
        {"key": "Team", "value": "support-eng"}
    ]
)

# Tag a knowledge base
bedrock.tag_resource(
    resourceARN=f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/kb-123",
    tags=[
        {"key": "Project", "value": "internal-docs"},
        {"key": "Team", "value": "documentation"}
    ]
)
""")

In [ ]:
# AWS Budgets — set up cost alerts for Bedrock spending
# This creates a budget that alerts when forecasted or actual spend exceeds the threshold
budgets = session.client("budgets")

budget_name = "genai-lab-bedrock-budget"

try:
    budgets.create_budget(
        AccountId=ACCOUNT_ID,
        Budget={
            "BudgetName": budget_name,
            "BudgetLimit": {"Amount": "50", "Unit": "USD"},
            "BudgetType": "COST",
            "TimeUnit": "MONTHLY",
            "CostFilters": {
                "Service": ["Amazon Bedrock"]
            }
        },
        NotificationsWithSubscribers=[{
            "Notification": {
                "NotificationType": "FORECASTED",
                "ComparisonOperator": "GREATER_THAN",
                "Threshold": 80.0,
                "ThresholdType": "PERCENTAGE"
            },
            "Subscribers": [{
                "SubscriptionType": "EMAIL",
                "Address": "admin@example.com"
            }]
        }]
    )
    print(f"Created budget: {budget_name}")
    print(f"  Monthly limit: $50 USD")
    print(f"  Service filter: Amazon Bedrock")
    print(f"  Alert: Email when forecasted spend exceeds 80% ($40)")
except budgets.exceptions.DuplicateRecordException:
    print(f"Budget '{budget_name}' already exists.")
except Exception as e:
    print(f"Could not create budget: {e}")
    print("(Budgets API may require specific IAM permissions.)")

print("\nBest practices for GenAI cost governance:")
print("  1. Set per-service budgets (Bedrock, S3, CloudWatch) separately")
print("  2. Use tag-based cost allocation to track spend by team/project")
print("  3. Enable Cost Explorer with hourly granularity for GenAI workloads")
print("  4. Monitor per-model spend — some models cost 10x more than others")
print("  5. Implement model tiering (use cheaper models for simple tasks)")

## Cleanup

Remove the resources created in this lab to avoid ongoing charges. The KMS key has a mandatory minimum 7-day waiting period before deletion.

In [ ]:
# 1. Schedule KMS key for deletion (minimum 7-day waiting period)
try:
    kms.schedule_key_deletion(KeyId=KEY_ID, PendingWindowInDays=7)
    print(f"KMS key {KEY_ID} scheduled for deletion in 7 days")
    print("  (Can be cancelled with kms.cancel_key_deletion() during the waiting period)")
except Exception as e:
    print(f"Could not schedule key deletion: {e}")

# Delete the KMS alias
try:
    kms.delete_alias(AliasName="alias/genai-lab-bedrock")
    print("Deleted alias: alias/genai-lab-bedrock")
except Exception as e:
    print(f"Could not delete alias: {e}")

# 2. Disable model invocation logging
try:
    bedrock.delete_model_invocation_logging_configuration()
    print("Model invocation logging DISABLED")
except Exception as e:
    print(f"Could not disable invocation logging: {e}")

# 3. Delete the budget
try:
    budgets.delete_budget(
        AccountId=ACCOUNT_ID,
        BudgetName=budget_name
    )
    print(f"Deleted budget: {budget_name}")
except Exception as e:
    print(f"Could not delete budget: {e}")

# 4. Delete CloudWatch log group (optional — contains invocation logs)
try:
    cloudwatch.delete_log_group(logGroupName="/aws/bedrock/model-invocations")
    print("Deleted log group: /aws/bedrock/model-invocations")
except Exception as e:
    print(f"Could not delete log group: {e}")

print("\nCleanup complete.")

## Key Takeaways

1. **Least-privilege IAM policies should be persona-based, granting only the Bedrock actions each role requires** — a developer needs only `InvokeModel`, a data scientist adds customization permissions, and only an admin should have `bedrock:*`, with all policies still scoped to the Bedrock service namespace
2. **VPC endpoints (PrivateLink) keep Bedrock traffic on the AWS private network, eliminating internet exposure** — this is a compliance requirement for regulated workloads (HIPAA, PCI, FedRAMP) and can be combined with VPC endpoint policies for fine-grained network-level access control
3. **KMS customer managed keys provide full control over encryption of Bedrock artifacts** — custom models, invocation logs, and knowledge base data can all be encrypted with your own keys, enabling key rotation policies, cross-account access control, and detailed audit trails of key usage
4. **CloudTrail and invocation logging serve different purposes and are both needed for comprehensive audit** — CloudTrail records API metadata (who called what, when, from where) while invocation logging captures the actual prompts and responses, and only invocation logging must be explicitly enabled
5. **Cost governance for GenAI requires tagging, budgets, and per-model spend awareness** — resource tags enable cost allocation by team and project, AWS Budgets provide proactive alerts, and model tiering (routing simple tasks to cheaper models) is the most impactful cost optimization lever

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Least Privilege | An IAM design principle where each identity receives only the minimum permissions required for its specific task, reducing blast radius if credentials are compromised — for Bedrock, this means scoping policies to specific actions like `InvokeModel` rather than `bedrock:*` |
| VPC Endpoint | A private connection between your VPC and an AWS service (like Bedrock) powered by AWS PrivateLink, allowing traffic to flow over the AWS private network without traversing the public internet — created per service (e.g., `bedrock-runtime`, `bedrock`, `bedrock-agent`) |
| PrivateLink | The AWS technology underlying interface VPC endpoints that provides private connectivity between VPCs and AWS services by placing elastic network interfaces (ENIs) in your subnets — enables `PrivateDnsEnabled` so existing code automatically routes through the private endpoint |
| KMS (Key Management Service) | AWS service for creating and managing encryption keys — customer managed keys (CMKs) can be used with Bedrock to encrypt custom models, invocation logs, and knowledge base data, with full control over key policies, rotation, and audit trails |
| CloudTrail | AWS audit logging service that records API calls across your account — captures Bedrock API metadata (who called what action, when, from which IP) but does NOT capture the content of prompts or model responses |
| Invocation Logging | A Bedrock-specific feature that captures the actual input prompts and model responses, delivered to CloudWatch Logs and/or S3 — must be explicitly enabled via `put_model_invocation_logging_configuration` and is separate from CloudTrail |
| Cost Tags | Key-value metadata attached to AWS resources for cost allocation and tracking — when activated in AWS Billing, tags like `Project`, `Team`, and `Environment` appear in Cost Explorer, enabling per-team chargeback and spend analysis for GenAI workloads |

## Exam Preparation — Common Exam Question Patterns

**Q: A company wants to ensure that Bedrock API calls from their application never traverse the public internet. What should they implement?**
A: Create an interface VPC endpoint for the `com.amazonaws.{region}.bedrock-runtime` service using AWS PrivateLink. Place the endpoint in the same VPC as the application, enable private DNS so existing code routes automatically through the private endpoint, and attach a VPC endpoint policy to restrict which Bedrock actions are permitted. This keeps all traffic on the AWS private network without requiring changes to application code.

**Q: How should you design IAM policies for a team where developers need to invoke models but only ML engineers should be able to fine-tune custom models?**
A: Create two separate IAM policies. The developer policy should allow only `bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream` scoped to `arn:aws:bedrock:{region}::foundation-model/*`. The ML engineer policy should include those same invoke actions plus `bedrock:CreateModelCustomizationJob`, `bedrock:GetModelCustomizationJob`, `bedrock:ListModelCustomizationJobs`, and `bedrock:GetCustomModel`. This follows least privilege by granting each role only the permissions it needs.

**Q: What is the difference between CloudTrail logging and Bedrock model invocation logging, and when would you use each?**
A: CloudTrail automatically records API call metadata — who made the call, what action was performed, when, and from which IP address — but does NOT capture prompt content or model responses. Model invocation logging must be explicitly enabled and captures the actual input prompts, model outputs, and token usage metrics, delivered to CloudWatch Logs and/or S3. Use CloudTrail for security auditing (detecting unauthorized access) and invocation logging for content auditing (reviewing what was sent to and returned from models).

**Q: A regulated financial services company needs to encrypt all data related to their Bedrock custom models. What encryption options are available?**
A: Bedrock encrypts all data at rest using AWS-managed keys by default, and all API calls are encrypted in transit with TLS 1.2+. For additional control, provide a customer managed KMS key (CMK) via the `customModelKmsKeyId` parameter when creating model customization jobs. This gives the company full control over key rotation, access policies (who can use the key), and audit trails (every use of the key is logged in CloudTrail). The same approach applies to knowledge bases (`serverSideEncryptionConfiguration`) and invocation logs (`s3EncryptionConfiguration`).

**Q: How can an organization track and control Bedrock costs across multiple teams?**
A: Implement a three-layer approach: (1) Apply consistent resource tags (`Project`, `Team`, `Environment`, `CostCenter`) to all Bedrock resources using `tag_resource`, then activate these as cost allocation tags in AWS Billing. (2) Create AWS Budgets filtered by the Amazon Bedrock service with notification thresholds (e.g., alert at 80% of monthly budget). (3) Use Cost Explorer with tag-based filtering to analyze per-team and per-model spend. Additionally, implement model tiering policies to route simple tasks to cheaper models, which is the most impactful cost optimization.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| KMS Key — Section C (created and scheduled for deletion) | 1 key, ~1 hour active | < $0.01 |
| KMS Key — Section C (scheduled deletion pending 7 days) | Key exists but disabled | $0.00 |
| Llama 3 8B — Section E (test invocation) | 1 call, ~100 tokens | < $0.01 |
| CloudWatch Logs — Section E (invocation log storage) | < 1 KB | < $0.01 |
| AWS Budgets — Section F (1 budget) | 1 budget | $0.00 (first 2 free) |
| CloudTrail — Section D (lookup events) | Read-only query | $0.00 |
| **Total** | | **< $1.00** |

This lab focuses on infrastructure configuration rather than model invocations, keeping costs minimal. The KMS key is the only resource with ongoing cost potential ($1/month if not deleted), but scheduling it for deletion in the cleanup section ensures it is removed after the 7-day mandatory waiting period. The single Llama 3 8B invocation in Section E costs a fraction of a cent.